This cell imports all required Python libraries for data processing, encoding, and scaling.

In [49]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
import os
import kagglehub

### Step 1 — Load the Dataset
We load the dataset to inspect its structure, number of rows/columns, and initial data types.


In [50]:



path = kagglehub.dataset_download("vikaseranki9/mobile-usage-behavioral-analysis-dataset")

print("Path to dataset files:", path)

csv_path = None
for file in os.listdir(path):
    if file.endswith(".csv"):
        csv_path = os.path.join(path, file)
        break

if csv_path is None:
    raise FileNotFoundError(f"No .csv file found in {path}")

df = pd.read_csv(csv_path)
df.head()


print("Initial shape:", df.shape)
print("\nInitial dtypes:\n", df.dtypes)


Path to dataset files: /Users/saivikas/.cache/kagglehub/datasets/vikaseranki9/mobile-usage-behavioral-analysis-dataset/versions/1
Initial shape: (1000, 10)

Initial dtypes:
 User_ID                           int64
Age                               int64
Gender                           object
Total_App_Usage_Hours           float64
Daily_Screen_Time_Hours         float64
Number_of_Apps_Used               int64
Social_Media_Usage_Hours        float64
Productivity_App_Usage_Hours    float64
Gaming_App_Usage_Hours          float64
Location                         object
dtype: object


### Step 2 — Missing Value Summary
We check how many missing values each column contains to determine if imputation is needed.


In [51]:
missing_summary = df.isna().sum().to_frame("missing_count")
missing_summary["missing_pct"] = (df.isna().mean() * 100).round(3)
print("\nMissing summary:\n", missing_summary)



Missing summary:
                               missing_count  missing_pct
User_ID                                   0          0.0
Age                                       0          0.0
Gender                                    0          0.0
Total_App_Usage_Hours                     0          0.0
Daily_Screen_Time_Hours                   0          0.0
Number_of_Apps_Used                       0          0.0
Social_Media_Usage_Hours                  0          0.0
Productivity_App_Usage_Hours              0          0.0
Gaming_App_Usage_Hours                    0          0.0
Location                                  0          0.0


### Step 3 — Datetime Parsing
We detect string columns that look like dates (contain '/', '-', ':') and convert them to proper datetime format.


In [52]:
df_work = df.copy()

for col in df_work.columns:
    if df_work[col].dtype == object:
        sample = df_work[col].dropna().astype(str).head(40)
        if len(sample) > 0:
            if sample.str.contains(r'[:T/-]').any():
                try:
                    parsed = pd.to_datetime(df_work[col], errors='coerce', infer_datetime_format=True)
                    if parsed.notna().sum() >= int(0.25 * df_work[col].notna().sum()):
                        df_work[col] = parsed
                        print(f"Parsed datetime column: {col}")
                except:
                    pass

print("\nDtypes after datetime parsing:\n", df_work.dtypes)



Dtypes after datetime parsing:
 User_ID                           int64
Age                               int64
Gender                           object
Total_App_Usage_Hours           float64
Daily_Screen_Time_Hours         float64
Number_of_Apps_Used               int64
Social_Media_Usage_Hours        float64
Productivity_App_Usage_Hours    float64
Gaming_App_Usage_Hours          float64
Location                         object
dtype: object


### Step 4 — Column Type Separation
We identify numeric, categorical, datetime, and boolean columns so each type can receive the correct preprocessing.


In [53]:
numeric_cols = df_work.select_dtypes(include=[np.number]).columns.tolist()
datetime_cols = df_work.select_dtypes(include=["datetime64"]).columns.tolist()
object_cols = df_work.select_dtypes(include=[object]).columns.tolist()
bool_cols = df_work.select_dtypes(include=["bool"]).columns.tolist()

print("\nNumeric columns:", numeric_cols)
print("Object columns:", object_cols)
print("Datetime columns:", datetime_cols)



Numeric columns: ['User_ID', 'Age', 'Total_App_Usage_Hours', 'Daily_Screen_Time_Hours', 'Number_of_Apps_Used', 'Social_Media_Usage_Hours', 'Productivity_App_Usage_Hours', 'Gaming_App_Usage_Hours']
Object columns: ['Gender', 'Location']
Datetime columns: []


### Step 5 — Handle Missing Values
We apply different strategies:
- Numeric → Median imputation  
- Categorical → Mode (most frequent) or `"Unknown"`
- Boolean → Fill with `False`


In [54]:
df_imputed = df_work.copy()

if numeric_cols:
    num_imputer = SimpleImputer(strategy="median")
    df_imputed[numeric_cols] = num_imputer.fit_transform(df_imputed[numeric_cols])

for col in object_cols:
    try:
        mode_value = df_imputed[col].mode(dropna=True).iloc[0]
    except:
        mode_value = "Unknown"
    df_imputed[col] = df_imputed[col].fillna(mode_value)

for col in bool_cols:
    df_imputed[col] = df_imputed[col].fillna(False)

print("\nMissing values handled.")



Missing values handled.


###  Step 6 — Remove Duplicate Rows
We delete exact duplicate rows to ensure cleaner data and avoid bias.


In [55]:
before = df_imputed.shape[0]
df_imputed = df_imputed.drop_duplicates().reset_index(drop=True)
after = df_imputed.shape[0]

print(f"\nDuplicates removed: {before - after}")



Duplicates removed: 0


###  Step 7 — Outlier Capping (IQR Winsorization)
Values beyond the range **[Q1 − 1.5×IQR, Q3 + 1.5×IQR]** are capped to reduce the effect of extreme outliers while preserving data.


In [56]:
outlier_caps = {}

for col in numeric_cols:
    if df_imputed[col].nunique() > 2:
        Q1 = df_imputed[col].quantile(0.25)
        Q3 = df_imputed[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        outlier_caps[col] = (lower, upper)
        df_imputed[col] = df_imputed[col].clip(lower=lower, upper=upper)

print("\nOutlier capping applied (IQR method).")



Outlier capping applied (IQR method).


###  Step 8 — Save Cleaned Datasets
We save:
- A cleaned dataset (human-readable)  
- A cleaned + encoded + scaled dataset (ML-ready)


In [57]:
cleaned_path = "mobile_usage_behavioral_analysis_cleaned.csv"

df_imputed.to_csv(cleaned_path, index=False)

print(f"\nSaved cleaned dataset → {cleaned_path}")



Saved cleaned dataset → mobile_usage_behavioral_analysis_cleaned.csv


###  Step 9 — Final Checks
We inspect the final shape and preview sample rows from the cleaned and scaled datasets.


In [58]:
print("\nFinal dataset shape cleaned:", df_imputed.shape)

print("\nSample of cleaned data:\n", df_imputed.head())



Final dataset shape cleaned: (1000, 10)

Sample of cleaned data:
    User_ID   Age  Gender  Total_App_Usage_Hours  Daily_Screen_Time_Hours  \
0      1.0  56.0    Male                   2.61                     7.15   
1      2.0  46.0    Male                   2.13                    13.79   
2      3.0  32.0  Female                   7.28                     4.50   
3      4.0  25.0  Female                   1.20                     6.29   
4      5.0  38.0    Male                   6.31                    12.59   

   Number_of_Apps_Used  Social_Media_Usage_Hours  \
0                 24.0                      4.43   
1                 18.0                      4.67   
2                 11.0                      4.58   
3                 21.0                      3.18   
4                 14.0                      3.15   

   Productivity_App_Usage_Hours  Gaming_App_Usage_Hours     Location  
0                          0.55                    2.40  Los Angeles  
1                    